# 01 - Frame explorer

**Goal.** Understand what the provided data says about *one moment* of the match before we build
anything. For a chosen (or random) video frame this notebook shows, together:

* the video frame itself, decoded from the tactical `.mp4`;
* the tracked players and the ball, projected onto that frame through the homography. If the
  markers land on the real players, the tracking, the homography and the video agree;
* the same tracking on a pitch diagram, in metres;
* a printed report: ball position / speed / how long it has been missing, every tracked slot, the
  homography quality, the Smart Tagging phases active at that frame, and the public ground-truth
  label when the frame is inside the public 5 minutes.

**Needs:** the match data in `data/` (see `DATA.md`), `ffmpeg` on the PATH, and
`numpy`, `pandas`, `matplotlib`, `Pillow`. No `kloppy`, no OpenCV.

**Rules that apply here.**
* The public ground-truth label is printed only to sanity-check our reading of the data. It must not
  be used to tune or train anything (`RULES.md`).
* Rendered frames go to `explore/output/`, which is git-ignored: a video frame is derived from the
  gated match data and must not reach the public repository.

Code lives in `frame_explorer.py` next to this notebook; `README.md` documents every assumption.

In [ ]:
import os
import sys

# Works whether Jupyter was started in the repo root or in explore/.
HERE = os.getcwd() if os.path.basename(os.getcwd()) == "explore" else os.path.join(os.getcwd(), "explore")
sys.path.insert(0, HERE)
sys.path.insert(0, os.path.dirname(HERE))

import matplotlib.pyplot as plt
import frame_explorer as fe

# Reads ~200 MB of tracking text once; takes a few seconds. Reuse `data` for every call below.
data = fe.MatchData()
print("video:", os.path.basename(data.video_path))
print("teams by slot, e.g. slot 1 ->", data.slot_team[1], "| slot 46 ->", data.slot_team[46])

## 1. A random frame

`seed` makes the draw reproducible, so a frame can be quoted in a report and found again.
Change the seed (or pass `seed=None`) for a different frame.

By default the draw is over **graded** frames only (the two halves). Look at the left image first:
do the coloured markers sit on the players? Then read the report below the figure.

In [ ]:
info = fe.explore(data, seed=42)

### How to read the output

| Part | What it tells you |
|---|---|
| Left image | The video frame. Blue = Barcelona (H), orange = Athletic (A), numbers = track **slots** (not shirt numbers), yellow ring = the tracked ball, cyan dashed = pitch edge from the homography. |
| Right pitch | The same tracking in metres. Orientation matches the camera: `y = 0` is the far touchline. |
| `BALL` lines | Position in metres, speed over the last 5 frames, and - when missing - how long since it was last seen and until it is seen again. |
| `homography` | `reprojection_error` is how far the fitted matrix misses; lower is better. |
| `TRACKED SLOTS` | One row per slot with a position (46 slots exist; about 20 to 22 are tracked during play). `dist_ball_m` is the distance to the tracked ball. `px, py` are image pixels. `frozen` = position identical to the previous frame (possibly a stale track). |
| `SMART TAGGING` | Automatic phases covering this frame. These are phases, not possessions, and the `Team` tag can be `N/A`. |

## 1b. The same frame as raw text

The figure and report above are *decoded* data. To see how the data actually arrives, print the
untouched source text behind a frame: the tracking line, the homography row, the Smart Tagging XML
and the ground-truth rows. Each block quotes a file verbatim; the `->` lines show the step to the
figure (normalised coordinates to metres, normalised coordinates through the homography to pixels,
slot number to team through the metadata, and so on).

Read it top to bottom once for frame 11500, then compare block [1] with the dots in the figure.

In [ ]:
raw_view = fe.format_raw(data, 11500)
print(raw_view)

## 2. A frame where the ball is *not* tracked

The ball is missing on about 41% of graded frames. Here the report shows how long the gap has lasted
and how long until the ball reappears. Note that the players are usually still tracked: that is the
information we have to use when the ball is gone.

In [ ]:
info = fe.explore(data, seed=7, ball="missing")

## 3. Context: does the ball go missing alone, or together with the players?

Top strip: is the ball tracked (green) or missing (red). Bottom line: how many of the 46 slots have a
position. If the two drop together, the tracker lost the scene; if only the ball drops, it was
probably occluded or out of play.

In [ ]:
fe.plot_context(data, info["frame"], seconds=30)
plt.show()

## 4. A frame inside the public ground-truth window (sanity check only)

Frames 11185 to 18684 are the first 5 minutes of play, the only part with a public label. Use it to
check that you read the data correctly (for example: is the nearest tracked player to the ball
usually from the team named in the label?). Do **not** use it to choose thresholds or train.

In [ ]:
info = fe.explore(data, frame=11500)

## 5. Your turn

Run the cells above again with other seeds, `half=2`, or `ball="present"`, and ask of every frame:

1. Do the markers land on the players in the video? (If not, which ones and why?)
2. Is the ball tracked? If not, where was it last seen and who was near it?
3. Is the nearest tracked player to the ball plausible as "the team in possession"?
4. What does Smart Tagging say, and does it agree with what you see?
5. Anything odd: a frozen slot, a player outside the pitch, a slot that jumped?

Write what you find in `docs/worklog.md` (date, frame number, what you saw), so reviewers can follow
the reasoning.

In [ ]:
# Re-run this cell for a new random frame. Add ball="present" / ball="missing" / half=1 / half=2.
info = fe.explore(data)